In [1]:
"""
Pure MLP listwise ranker for AFAC Task 2.

Fourth-generation balanced rebound version:
1. Pure PyTorch MLP only.
2. True tail-qid meta validation using original train row index.
3. ApproxRank / NDCG-aware loss.
4. Hard negative mining.
5. Anti-popularity candidate reweighting.
6. Stronger leakage-prone feature pruning.
7. Conservative final refit epoch cap.

Output:
A2_mlp_v4_balanced.csv
"""

from __future__ import annotations

import copy
import gc
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import combinations
from typing import Any, Dict, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------ Configuration ------------------------------

DATA_ROOT = "/kaggle/input/datasets/theunforgiven7/afac-task2/A�Ƽ�"
OUTPUT_DIR = "/kaggle/working"

OUTPUT_NAMES = {
    "balanced": "A2_mlp_v4_balanced.csv",
    "safe": "A2_mlp_v4_safe.csv",
    "pure_mlp": "A2_mlp_v4_pure_mlp.csv",
}

N_FOLDS = 5
RANDOM_STATE = 2026
MAX_CANDIDATES = 170
RECENT_ITEMS = 8
MAX_PAIR_FEATURE_COLS = 6

TOP_GLOBAL = 80
TOP_LAST1 = 72
TOP_LAST2 = 62
TOP_LAST3 = 52
TOP_RECENT_EACH = 28
TOP_TRANSITION = 42
TOP_PROFILE = 30
TOP_SEGMENT = 22
TOP_SEGMENT_PAIR = 16
TOP_LENGTH_BUCKET = 25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MLP_EPOCHS = 28
MLP_PATIENCE = 4
MLP_QUERY_BATCH = 56
MLP_LR = 6e-4
MLP_WEIGHT_DECAY = 1.6e-3
MLP_DROPOUT = 0.31
MLP_HIDDEN = (256, 128, 64)

MLP_TRAIN_NATURAL_ONLY = True

MLP_AUX_BCE_WEIGHT = 0.012
MLP_PAIRWISE_WEIGHT = 0.040
MLP_APPROX_NDCG_WEIGHT = 0.300
MLP_HARD_NEG_WEIGHT = 0.120

APPROX_RANK_TAU = 1.0
HARD_NEG_K = 6
HARD_NEG_MARGIN = 0.18

ANTI_POPULARITY_ENABLED = True
ANTI_POPULARITY_POWER = 0.08

META_VALID_FRACTION = 0.20

# Anti-overfit training noise and production-time robust blending.
# The blend keeps the neural ranker from over-trusting fold-specific tail statistics.
MLP_INPUT_NOISE_STD = 0.008
MLP_FEATURE_DROPOUT = 0.015

ROBUST_BLEND_MLP_WEIGHT = 0.78
ROBUST_BLEND_RRF_WEIGHT = 0.12
ROBUST_BLEND_SUFFIX_WEIGHT = 0.04
ROBUST_BLEND_RECENCY_WEIGHT = 0.04
ROBUST_BLEND_SOURCE_WEIGHT = 0.02
ROBUST_SCORE_COL = "blend_score"
SAFE_BLEND_SCORE_COL = "blend_safe_score"

# These are converted into query-wise percentile rank features.
# Keep only relatively stable behavior / sequence features.
MLP_QUERY_RANK_FEATURES = [
    "rrf",
    "source_count",
    "best_rank_recip",
    "history_present",
    "history_count",
    "history_recency",
    "is_last_raw",
    "is_last_dedup",
    "last1_prob",
    "last2_prob",
    "last3_prob",
    "last1_lift",
    "last2_lift",
    "last3_lift",
    "recent_prob_sum",
    "recent_prob_max",
    "recent_lift_sum",
    "recent_lift_max",
    "recent_hit_sources",
    "transition_prob",
    "transition_lift",
    "length_prob",
    "suffix_best_prob",
    "suffix_best_lift",
    "repeat_x_recent",
    "last1_x_history",
]

# Real MLP base-feature pruning.
# Previous version only removed rank features; this version removes raw high-risk features too.
DROP_MLP_FEATURES_EXACT = {
    "global_target_logcnt",
    "global_target_prior",
    "global_target_logprior",
    "global_target_rank_recip",
    "sequence_item_logcnt",
    "sequence_item_prior",
    "profile_logcnt",
    "profile_logsupport",
    "profile_prob",
    "profile_lift",
    "profile_mle",
    "seg_prob_max",
    "seg_prob_mean",
    "seg_lift_max",
    "seg_lift_mean",
    "seg_logcnt_sum",
    "pair_prob_max",
    "pair_prob_mean",
    "pair_lift_max",
    "pair_lift_mean",
    "pair_logcnt_sum",
    "transition_mle",
    "last1_mle",
    "last2_mle",
    "last3_mle",
    "length_mle",
    "last1_logsupport",
    "last2_logsupport",
    "last3_logsupport",
    "transition_logsupport",
    "length_logsupport",
}

DROP_MLP_FEATURE_PREFIXES = (
    "profile_",
    "pair_",
)

DROP_MLP_FEATURE_SUFFIXES = (
    "_mle",
)


# ------------------------------- Data helpers -------------------------------


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


def find_data_base(root: str) -> str:
    required = {"train.csv", "test.csv", "user.csv", "sample_submission.csv"}

    if os.path.isdir(root):
        names = set(os.listdir(root))
        if required.issubset(names):
            return root

    for current, _, files in os.walk(root):
        if required.issubset(set(files)):
            return current

    raise FileNotFoundError(
        f"Could not find {sorted(required)} under {root!r}. "
        "Please set DATA_ROOT to the correct Kaggle dataset directory."
    )


def load_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    base = find_data_base(DATA_ROOT)
    print(f"Data directory: {base}")

    train = pd.read_csv(os.path.join(base, "train.csv"))
    test = pd.read_csv(os.path.join(base, "test.csv"))
    user = pd.read_csv(os.path.join(base, "user.csv"))
    sample = pd.read_csv(os.path.join(base, "sample_submission.csv"))

    return train, test, user, sample


def clean_scalar(x: Any) -> str:
    if pd.isna(x):
        return "__NA__"

    s = str(x).strip()
    return s if s and s.lower() != "nan" else "__NA__"


def clean_id(x: Any) -> str:
    if pd.isna(x):
        return ""

    if isinstance(x, (np.integer, int)):
        return str(int(x))

    if isinstance(x, (np.floating, float)) and float(x).is_integer():
        return str(int(x))

    s = str(x).strip()
    return "" if s.lower() == "nan" else s


def split_items(x: Any) -> List[str]:
    if pd.isna(x):
        return []

    s = str(x).strip()

    if not s or s.lower() == "nan":
        return []

    return [clean_id(v) for v in s.split(",") if clean_id(v)]


def dedup_preserve_order(items: Sequence[str]) -> List[str]:
    seen = set()
    out: List[str] = []

    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)

    return out


def length_bucket(n: int) -> str:
    if n <= 0:
        return "0"
    if n == 1:
        return "1"
    if n == 2:
        return "2"
    if n <= 4:
        return "3-4"
    if n <= 7:
        return "5-7"
    if n <= 15:
        return "8-15"
    if n <= 30:
        return "16-30"

    return "31+"


def prepare_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    user: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    for df in (train, test, user):
        if "uid" not in df.columns:
            raise KeyError("Every data file must contain a 'uid' column.")
        df["uid"] = df["uid"].map(clean_id)

    train["target_iid"] = train["target_iid"].map(clean_id)

    cat_cols = [c for c in user.columns if c.startswith("u_cat_")]

    for col in cat_cols:
        user[col] = user[col].map(clean_scalar)

    train = train.merge(
        user[["uid"] + cat_cols],
        on="uid",
        how="left",
        validate="many_to_one",
    )

    test = test.merge(
        user[["uid"] + cat_cols],
        on="uid",
        how="left",
        validate="many_to_one",
    )

    for df in (train, test):
        for col in cat_cols:
            df[col] = df[col].map(clean_scalar)

    return train, test, cat_cols


# ---------------------------- Statistics / recall ---------------------------


@dataclass
class CandidateMeta:
    rrf: float = 0.0
    source_count: int = 0
    best_rank: int = 10_000


class RecallStats:
    def __init__(self, df: pd.DataFrame, cat_cols: Sequence[str]):
        self.cat_cols = list(cat_cols)
        self.pair_cols = list(combinations(self.cat_cols[:MAX_PAIR_FEATURE_COLS], 2))

        self.n_rows = max(len(df), 1)
        self.target_counts: Counter[str] = Counter()
        self.sequence_item_counts: Counter[str] = Counter()

        self.last1_to_target: MutableMapping[str, Counter[str]] = defaultdict(Counter)
        self.last2_to_target: MutableMapping[Tuple[str, ...], Counter[str]] = defaultdict(Counter)
        self.last3_to_target: MutableMapping[Tuple[str, ...], Counter[str]] = defaultdict(Counter)
        self.recent_item_to_target: MutableMapping[str, Counter[str]] = defaultdict(Counter)
        self.transition: MutableMapping[str, Counter[str]] = defaultdict(Counter)

        self.segment_to_target: Dict[str, MutableMapping[str, Counter[str]]] = {
            c: defaultdict(Counter) for c in self.cat_cols
        }

        self.segment_pair_to_target: Dict[
            Tuple[str, str],
            MutableMapping[Tuple[str, str], Counter[str]],
        ] = {pair: defaultdict(Counter) for pair in self.pair_cols}

        self.profile_to_target: MutableMapping[Tuple[str, ...], Counter[str]] = defaultdict(Counter)
        self.length_to_target: MutableMapping[str, Counter[str]] = defaultdict(Counter)

        self._build(df)

        self.target_total = float(sum(self.target_counts.values())) or 1.0
        self.seq_item_total = float(sum(self.sequence_item_counts.values())) or 1.0
        self.n_targets = max(len(self.target_counts), 1)

        self.global_top = [iid for iid, _ in self.target_counts.most_common(TOP_GLOBAL)]
        self.target_rank = {
            iid: rank
            for rank, (iid, _) in enumerate(self.target_counts.most_common(), start=1)
        }

    def _build(self, df: pd.DataFrame) -> None:
        columns = list(df.columns)

        for values in df.itertuples(index=False, name=None):
            row = dict(zip(columns, values))
            target = clean_id(row.get("target_iid"))

            if not target:
                continue

            raw = split_items(row.get("item_seq_raw"))
            dedup = split_items(row.get("item_seq_dedup"))

            if not dedup:
                dedup = dedup_preserve_order(raw)

            self.target_counts[target] += 1.0
            self.sequence_item_counts.update(raw)

            if dedup:
                self.last1_to_target[dedup[-1]][target] += 1.0

                if len(dedup) >= 2:
                    self.last2_to_target[tuple(dedup[-2:])][target] += 1.0

                if len(dedup) >= 3:
                    self.last3_to_target[tuple(dedup[-3:])][target] += 1.0

                for distance, item in enumerate(reversed(dedup[-RECENT_ITEMS:])):
                    self.recent_item_to_target[item][target] += 0.82 ** distance

            for left, right in zip(raw[:-1], raw[1:]):
                self.transition[left][right] += 1.0

            for col in self.cat_cols:
                val = clean_scalar(row.get(col))
                self.segment_to_target[col][val][target] += 1.0

            for c1, c2 in self.pair_cols:
                key = (clean_scalar(row.get(c1)), clean_scalar(row.get(c2)))
                self.segment_pair_to_target[(c1, c2)][key][target] += 1.0

            profile = tuple(clean_scalar(row.get(c)) for c in self.cat_cols)
            self.profile_to_target[profile][target] += 1.0
            self.length_to_target[length_bucket(len(raw))][target] += 1.0

    def _popularity_penalty(self, iid: str) -> float:
        if not ANTI_POPULARITY_ENABLED:
            return 1.0

        cnt = float(self.target_counts.get(iid, 0.0))

        # Mild anti-popularity penalty.
        # Count=0 -> 1.0; very popular items get only a small downweight.
        base = math.log1p(2.0)
        denom = math.log1p(cnt + 2.0)

        return float((base / max(denom, 1e-6)) ** ANTI_POPULARITY_POWER)

    def _add_source(
        self,
        meta: MutableMapping[str, CandidateMeta],
        counter: Mapping[str, float],
        topn: int,
        source_weight: float,
        anti_pop: bool = True,
    ) -> None:
        if not counter:
            return

        if isinstance(counter, Counter):
            ranked = counter.most_common(topn)
        else:
            ranked = sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))[:topn]

        for rank, (iid, _) in enumerate(ranked, start=1):
            penalty = self._popularity_penalty(iid) if anti_pop else 1.0

            current = meta[iid]
            current.rrf += (source_weight / (10.0 + rank)) * penalty
            current.source_count += 1
            current.best_rank = min(current.best_rank, rank)

    def generate_candidates(
        self,
        row: Mapping[str, Any],
        max_candidates: int = MAX_CANDIDATES,
    ) -> Tuple[List[str], Dict[str, CandidateMeta]]:
        raw = split_items(row.get("item_seq_raw"))
        dedup = split_items(row.get("item_seq_dedup"))

        if not dedup:
            dedup = dedup_preserve_order(raw)

        meta: Dict[str, CandidateMeta] = defaultdict(CandidateMeta)

        history_score: Counter[str] = Counter()
        raw_counts = Counter(raw)

        for iid, cnt in raw_counts.items():
            history_score[iid] += 2.0 * math.log1p(cnt)

        for distance, iid in enumerate(reversed(raw)):
            history_score[iid] += 3.0 / (1.0 + distance)

        self._add_source(meta, history_score, min(len(history_score), 60), 7.0, anti_pop=False)

        if dedup:
            last1 = dedup[-1]

            self._add_source(meta, self.last1_to_target.get(last1, {}), TOP_LAST1, 6.0)
            self._add_source(meta, self.transition.get(last1, {}), TOP_TRANSITION, 2.0)

            if len(dedup) >= 2:
                self._add_source(
                    meta,
                    self.last2_to_target.get(tuple(dedup[-2:]), {}),
                    TOP_LAST2,
                    8.0,
                )

            if len(dedup) >= 3:
                self._add_source(
                    meta,
                    self.last3_to_target.get(tuple(dedup[-3:]), {}),
                    TOP_LAST3,
                    10.0,
                )

            for distance, item in enumerate(reversed(dedup[-RECENT_ITEMS:])):
                self._add_source(
                    meta,
                    self.recent_item_to_target.get(item, {}),
                    TOP_RECENT_EACH,
                    4.0 * (0.82 ** distance),
                )

        profile = tuple(clean_scalar(row.get(c)) for c in self.cat_cols)
        self._add_source(meta, self.profile_to_target.get(profile, {}), TOP_PROFILE, 2.4)

        for col in self.cat_cols:
            val = clean_scalar(row.get(col))
            self._add_source(meta, self.segment_to_target[col].get(val, {}), TOP_SEGMENT, 1.25)

        for c1, c2 in self.pair_cols:
            key = (clean_scalar(row.get(c1)), clean_scalar(row.get(c2)))
            self._add_source(
                meta,
                self.segment_pair_to_target[(c1, c2)].get(key, {}),
                TOP_SEGMENT_PAIR,
                1.10,
            )

        self._add_source(
            meta,
            self.length_to_target.get(length_bucket(len(raw)), {}),
            TOP_LENGTH_BUCKET,
            0.8,
        )

        self._add_source(meta, self.target_counts, TOP_GLOBAL, 0.45)

        ranked = sorted(
            meta,
            key=lambda iid: (
                -meta[iid].rrf,
                -meta[iid].source_count,
                meta[iid].best_rank,
                iid,
            ),
        )[:max_candidates]

        return ranked, dict(meta)

    def target_prior(self, iid: str) -> float:
        return (self.target_counts.get(iid, 0.0) + 0.5) / (
            self.target_total + 0.5 * self.n_targets
        )

    def sequence_prior(self, iid: str) -> float:
        return (self.sequence_item_counts.get(iid, 0.0) + 0.5) / (
            self.seq_item_total + 0.5 * max(len(self.sequence_item_counts), 1)
        )

    def conditional_features(
        self,
        counter: Mapping[str, float],
        iid: str,
        alpha: float,
        prior: Optional[float] = None,
    ) -> Tuple[float, float, float, float, float]:
        prior = self.target_prior(iid) if prior is None else prior

        count = float(counter.get(iid, 0.0)) if counter else 0.0
        total = float(sum(counter.values())) if counter else 0.0

        p = (count + alpha * prior) / (total + alpha) if total + alpha > 0 else prior
        lift = math.log((p + 1e-12) / (prior + 1e-12))

        return math.log1p(count), math.log1p(total), p, lift, count / (total + 1e-12)

    def pair_features(
        self,
        row: Mapping[str, Any],
        iid: str,
        meta: CandidateMeta,
    ) -> Dict[str, float]:
        raw = split_items(row.get("item_seq_raw"))
        dedup = split_items(row.get("item_seq_dedup"))

        if not dedup:
            dedup = dedup_preserve_order(raw)

        raw_counts = Counter(raw)

        reverse_position: Dict[str, int] = {}
        for distance, item in enumerate(reversed(raw)):
            reverse_position.setdefault(item, distance)

        seq_len = len(raw)
        dedup_len = len(dedup)
        prior = self.target_prior(iid)
        seq_prior = self.sequence_prior(iid)

        features: Dict[str, float] = {
            "rrf": float(meta.rrf),
            "source_count": float(meta.source_count),
            "best_rank_recip": 0.0 if meta.best_rank >= 10_000 else 1.0 / meta.best_rank,
            "seq_len": float(seq_len),
            "log_seq_len": math.log1p(seq_len),
            "dedup_len": float(dedup_len),
            "unique_ratio": dedup_len / max(seq_len, 1),
            "is_cold_0": float(seq_len == 0),
            "is_cold_le2": float(seq_len <= 2),
            "global_target_logcnt": math.log1p(self.target_counts.get(iid, 0.0)),
            "global_target_prior": prior,
            "global_target_logprior": math.log(prior + 1e-12),
            "global_target_rank_recip": 1.0 / self.target_rank.get(iid, self.n_targets + 1),
            "sequence_item_logcnt": math.log1p(self.sequence_item_counts.get(iid, 0.0)),
            "sequence_item_prior": seq_prior,
            "history_present": float(iid in raw_counts),
            "history_count": float(raw_counts.get(iid, 0)),
            "history_logcount": math.log1p(raw_counts.get(iid, 0)),
            "history_freq_ratio": raw_counts.get(iid, 0) / max(seq_len, 1),
            "history_recency": 0.0
            if iid not in reverse_position
            else 1.0 / (1.0 + reverse_position[iid]),
            "is_last_raw": float(bool(raw) and iid == raw[-1]),
            "is_last_dedup": float(bool(dedup) and iid == dedup[-1]),
            "is_second_last": float(len(dedup) >= 2 and iid == dedup[-2]),
            "is_third_last": float(len(dedup) >= 3 and iid == dedup[-3]),
        }

        suffix_specs: List[Tuple[str, Mapping[str, float], float]] = []

        if dedup:
            suffix_specs.append(("last1", self.last1_to_target.get(dedup[-1], {}), 15.0))
        else:
            suffix_specs.append(("last1", {}, 15.0))

        if len(dedup) >= 2:
            suffix_specs.append(("last2", self.last2_to_target.get(tuple(dedup[-2:]), {}), 8.0))
        else:
            suffix_specs.append(("last2", {}, 8.0))

        if len(dedup) >= 3:
            suffix_specs.append(("last3", self.last3_to_target.get(tuple(dedup[-3:]), {}), 4.0))
        else:
            suffix_specs.append(("last3", {}, 4.0))

        for prefix, counter, alpha in suffix_specs:
            cnt, support, prob, lift, mle = self.conditional_features(counter, iid, alpha)
            features[f"{prefix}_logcnt"] = cnt
            features[f"{prefix}_logsupport"] = support
            features[f"{prefix}_prob"] = prob
            features[f"{prefix}_lift"] = lift
            features[f"{prefix}_mle"] = mle

        recent_prob_sum = 0.0
        recent_prob_max = 0.0
        recent_lift_sum = 0.0
        recent_lift_max = -50.0
        recent_count_sum = 0.0
        recent_hit_sources = 0.0

        for distance, item in enumerate(reversed(dedup[-RECENT_ITEMS:])):
            counter = self.recent_item_to_target.get(item, {})
            cnt, _, prob, lift, _ = self.conditional_features(counter, iid, 12.0)
            weight = 0.82 ** distance

            recent_prob_sum += weight * prob
            recent_prob_max = max(recent_prob_max, prob)
            recent_lift_sum += weight * lift
            recent_lift_max = max(recent_lift_max, lift)
            recent_count_sum += weight * cnt
            recent_hit_sources += float(counter.get(iid, 0.0) > 0)

        features["recent_prob_sum"] = recent_prob_sum
        features["recent_prob_max"] = recent_prob_max
        features["recent_lift_sum"] = recent_lift_sum
        features["recent_lift_max"] = recent_lift_max if dedup else 0.0
        features["recent_logcount_sum"] = recent_count_sum
        features["recent_hit_sources"] = recent_hit_sources

        transition_counter = self.transition.get(dedup[-1], {}) if dedup else {}
        tcnt, tsupport, tprob, tlift, tmle = self.conditional_features(
            transition_counter,
            iid,
            15.0,
            prior=seq_prior,
        )

        features.update(
            {
                "transition_logcnt": tcnt,
                "transition_logsupport": tsupport,
                "transition_prob": tprob,
                "transition_lift": tlift,
                "transition_mle": tmle,
            }
        )

        profile = tuple(clean_scalar(row.get(c)) for c in self.cat_cols)
        pcnt, psupport, pprob, plift, pmle = self.conditional_features(
            self.profile_to_target.get(profile, {}),
            iid,
            10.0,
        )

        features.update(
            {
                "profile_logcnt": pcnt,
                "profile_logsupport": psupport,
                "profile_prob": pprob,
                "profile_lift": plift,
                "profile_mle": pmle,
            }
        )

        lcnt, lsupport, lprob, llift, lmle = self.conditional_features(
            self.length_to_target.get(length_bucket(seq_len), {}),
            iid,
            30.0,
        )

        features.update(
            {
                "length_logcnt": lcnt,
                "length_logsupport": lsupport,
                "length_prob": lprob,
                "length_lift": llift,
                "length_mle": lmle,
            }
        )

        seg_probs: List[float] = []
        seg_lifts: List[float] = []
        seg_counts: List[float] = []

        for col in self.cat_cols:
            val = clean_scalar(row.get(col))
            cnt, _, prob, lift, _ = self.conditional_features(
                self.segment_to_target[col].get(val, {}),
                iid,
                25.0,
            )

            features[f"seg_{col}_logcnt"] = cnt
            features[f"seg_{col}_prob"] = prob
            features[f"seg_{col}_lift"] = lift

            seg_probs.append(prob)
            seg_lifts.append(lift)
            seg_counts.append(cnt)

        pair_probs: List[float] = []
        pair_lifts: List[float] = []
        pair_counts: List[float] = []

        for c1, c2 in self.pair_cols:
            key = (clean_scalar(row.get(c1)), clean_scalar(row.get(c2)))
            cnt, _, prob, lift, _ = self.conditional_features(
                self.segment_pair_to_target[(c1, c2)].get(key, {}),
                iid,
                15.0,
            )

            pair_probs.append(prob)
            pair_lifts.append(lift)
            pair_counts.append(cnt)

        features["seg_prob_max"] = max(seg_probs, default=prior)
        features["seg_prob_mean"] = float(np.mean(seg_probs)) if seg_probs else prior
        features["seg_lift_max"] = max(seg_lifts, default=0.0)
        features["seg_lift_mean"] = float(np.mean(seg_lifts)) if seg_lifts else 0.0
        features["seg_logcnt_sum"] = float(sum(seg_counts))

        features["pair_prob_max"] = max(pair_probs, default=prior)
        features["pair_prob_mean"] = float(np.mean(pair_probs)) if pair_probs else prior
        features["pair_lift_max"] = max(pair_lifts, default=0.0)
        features["pair_lift_mean"] = float(np.mean(pair_lifts)) if pair_lifts else 0.0
        features["pair_logcnt_sum"] = float(sum(pair_counts))

        features["repeat_x_recent"] = features["history_present"] * features["history_recency"]
        features["last1_x_history"] = features["last1_prob"] * (1.0 + features["history_logcount"])

        features["suffix_best_prob"] = max(
            features["last1_prob"],
            features["last2_prob"],
            features["last3_prob"],
        )

        features["suffix_best_lift"] = max(
            features["last1_lift"],
            features["last2_lift"],
            features["last3_lift"],
        )

        return features


# ------------------------ OOF feature generation ----------------------------


def make_random_group_folds(
    groups: Sequence[str],
    n_splits: int,
    seed: int,
) -> np.ndarray:
    unique_groups = np.array(pd.unique(pd.Series(groups)), dtype=object)

    if len(unique_groups) < 2:
        raise ValueError("At least two unique uid values are required for validation.")

    n_splits = min(n_splits, len(unique_groups))

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)

    group_to_fold = {g: i % n_splits for i, g in enumerate(unique_groups)}

    return np.array([group_to_fold[g] for g in groups], dtype=np.int16)


def build_candidate_frame(
    df: pd.DataFrame,
    stats: RecallStats,
    qid_start: int,
    include_label: bool,
    force_positive: bool,
) -> Tuple[pd.DataFrame, int, float]:
    rows: List[Dict[str, Any]] = []
    retrieved_positive = 0

    columns = list(df.columns)
    use_orig_qid = "_orig_qid" in df.columns

    for local_qid, values in enumerate(df.itertuples(index=False, name=None)):
        row = dict(zip(columns, values))

        if use_orig_qid:
            qid = int(row["_orig_qid"])
        else:
            qid = qid_start + local_qid

        candidates, meta = stats.generate_candidates(row)

        target = clean_id(row.get("target_iid")) if include_label else ""
        was_retrieved = bool(target and target in candidates)
        retrieved_positive += int(was_retrieved)

        if include_label and force_positive and target and target not in candidates:
            candidates.append(target)
            meta[target] = CandidateMeta()

        if len(candidates) > MAX_CANDIDATES:
            if include_label and target in candidates[MAX_CANDIDATES:]:
                candidates = candidates[: MAX_CANDIDATES - 1] + [target]
            else:
                candidates = candidates[:MAX_CANDIDATES]

        for iid in candidates:
            feats = stats.pair_features(row, iid, meta.get(iid, CandidateMeta()))
            feats["qid"] = qid
            feats["uid"] = clean_id(row.get("uid"))
            feats["candidate_iid"] = iid

            if include_label:
                feats["label"] = int(iid == target)
                feats["positive_retrieved"] = int(was_retrieved)

            rows.append(feats)

    frame = pd.DataFrame(rows)
    recall = retrieved_positive / max(len(df), 1) if include_label else float("nan")

    next_qid = qid_start if use_orig_qid else qid_start + len(df)

    return frame, next_qid, recall


def build_oof_training_frame(
    train: pd.DataFrame,
    cat_cols: Sequence[str],
) -> Tuple[pd.DataFrame, List[float]]:
    work = train.copy()
    work["_orig_qid"] = np.arange(len(work), dtype=np.int64)

    fold_ids = make_random_group_folds(work["uid"].tolist(), N_FOLDS, RANDOM_STATE)

    all_frames: List[pd.DataFrame] = []
    fold_recalls: List[float] = []
    qid_start = 0

    unique_folds = sorted(np.unique(fold_ids))

    for fold in unique_folds:
        fit_df = work.loc[fold_ids != fold].reset_index(drop=True)

        # Keep original train order inside valid fold.
        valid_df = (
            work.loc[fold_ids == fold]
            .sort_values("_orig_qid", kind="stable")
            .reset_index(drop=True)
        )

        print(
            f"\nFold {fold + 1}/{len(unique_folds)}: "
            f"fit={len(fit_df):,}, valid={len(valid_df):,}"
        )

        stats = RecallStats(fit_df, cat_cols)

        fold_frame, qid_start, recall = build_candidate_frame(
            valid_df,
            stats,
            qid_start=qid_start,
            include_label=True,
            force_positive=True,
        )

        print(f"  leakage-free candidate recall@{MAX_CANDIDATES}: {recall:.4f}")

        fold_recalls.append(recall)
        all_frames.append(fold_frame)

        del stats, fold_frame, fit_df, valid_df
        gc.collect()

    oof = pd.concat(all_frames, ignore_index=True)

    return oof, fold_recalls


def ndcg_hit_from_ranked_frame(
    frame: pd.DataFrame,
    score_col: str,
) -> Tuple[float, float]:
    ndcgs: List[float] = []
    hits: List[float] = []

    for _, group in frame.groupby("qid", sort=False):
        if int(group["positive_retrieved"].iloc[0]) == 0:
            ndcgs.append(0.0)
            hits.append(0.0)
            continue

        ranked = group.sort_values(score_col, ascending=False, kind="stable").head(10)
        labels = ranked["label"].to_numpy()

        hit_positions = np.flatnonzero(labels == 1)

        if len(hit_positions):
            rank = int(hit_positions[0])
            ndcgs.append(1.0 / math.log2(rank + 2.0))
            hits.append(1.0)
        else:
            ndcgs.append(0.0)
            hits.append(0.0)

    return float(np.mean(ndcgs)), float(np.mean(hits))


def split_qids_for_meta_validation(
    frame: pd.DataFrame,
    valid_fraction: float = META_VALID_FRACTION,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    True tail-qid validation.

    In v1, qid was generated after fold concatenation, so tail validation was not
    necessarily true train-tail validation. Here qid equals original train row index,
    so this split really takes the latest tail part of the original training order.
    """
    qids = np.sort(frame["qid"].drop_duplicates().to_numpy())
    n_valid = max(1, int(round(len(qids) * valid_fraction)))

    valid_qids = set(qids[-n_valid:])
    is_valid = frame["qid"].isin(valid_qids)

    train_part = (
        frame.loc[~is_valid]
        .sort_values("qid", kind="stable")
        .reset_index(drop=True)
    )

    valid_part = (
        frame.loc[is_valid]
        .sort_values("qid", kind="stable")
        .reset_index(drop=True)
    )

    return train_part, valid_part


def is_dropped_mlp_feature(col: str) -> bool:
    if col in DROP_MLP_FEATURES_EXACT:
        return True

    if any(col.startswith(prefix) for prefix in DROP_MLP_FEATURE_PREFIXES):
        return True

    if any(col.endswith(suffix) for suffix in DROP_MLP_FEATURE_SUFFIXES):
        return True

    return False


def feature_columns(frame: pd.DataFrame) -> List[str]:
    ignore_cols = {
        "qid",
        "uid",
        "candidate_iid",
        "label",
        "positive_retrieved",
    }

    cols = []

    for c in frame.columns:
        if c in ignore_cols:
            continue

        if is_dropped_mlp_feature(c):
            continue

        cols.append(c)

    return cols


def feature_matrix(frame: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    return (
        frame.reindex(columns=cols, fill_value=0.0)
        .replace([np.inf, -np.inf], 0.0)
        .fillna(0.0)
        .astype(np.float32)
    )


def natural_recall_subset(frame: pd.DataFrame) -> pd.DataFrame:
    good_qids = frame.loc[
        frame["positive_retrieved"].eq(1),
        "qid",
    ].drop_duplicates()

    out = (
        frame.loc[frame["qid"].isin(good_qids)]
        .sort_values("qid", kind="stable")
        .reset_index(drop=True)
    )

    if out.empty:
        return frame.sort_values("qid", kind="stable").reset_index(drop=True)

    return out


# ----------------------------- Pure MLP ranker ------------------------------


def selected_mlp_rank_cols(base_cols: Sequence[str]) -> List[str]:
    base_set = set(base_cols)
    return [c for c in MLP_QUERY_RANK_FEATURES if c in base_set]


def mlp_feature_matrix(
    frame: pd.DataFrame,
    base_cols: Sequence[str],
    rank_cols: Sequence[str],
) -> np.ndarray:
    base = feature_matrix(frame, base_cols).to_numpy(dtype=np.float32, copy=False)

    if not rank_cols:
        return base.copy()

    rank_arrays: List[np.ndarray] = []

    for col in rank_cols:
        rank_arr = (
            frame.groupby("qid", sort=False)[col]
            .rank(method="average", pct=True, ascending=True)
            .fillna(0.5)
            .to_numpy(dtype=np.float32)
        )

        rank_arrays.append(rank_arr)

    rank_mat = np.column_stack(rank_arrays).astype(np.float32, copy=False)

    return np.concatenate([base, rank_mat], axis=1)


def fit_standardizer(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    mean = X.mean(axis=0).astype(np.float32)
    std = X.std(axis=0).astype(np.float32)

    std[std < 1e-6] = 1.0

    return mean, std


def standardize_features(
    X: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    X = np.nan_to_num(
        X.astype(np.float32, copy=False),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    X = (X - mean) / std
    X = np.clip(X, -8.0, 8.0)

    return X.astype(np.float32, copy=False)


def make_query_slices(frame: pd.DataFrame) -> List[Tuple[int, int]]:
    sizes = frame.groupby("qid", sort=False).size().to_numpy(dtype=np.int64)
    ends = np.cumsum(sizes)
    starts = ends - sizes

    return list(zip(starts.tolist(), ends.tolist()))


class MLPListwiseRanker(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden: Sequence[int] = MLP_HIDDEN,
        dropout: float = MLP_DROPOUT,
    ) -> None:
        super().__init__()

        layers: List[nn.Module] = []
        prev = input_dim

        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.LayerNorm(h))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(dropout))
            prev = h

        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)

        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="linear")

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x)).squeeze(-1)


def approx_ndcg_loss_from_logits(
    logits: torch.Tensor,
    pos_idx: int,
    tau: float = APPROX_RANK_TAU,
) -> torch.Tensor:
    """
    Differentiable approximation of 1 - NDCG for one positive item.

    Approx rank:
        rank ~= 1 + sum(sigmoid((neg_score - pos_score) / tau))

    NDCG for single positive:
        1 / log2(rank + 1)

    This uses model scores, not original candidate order.
    """
    n = logits.numel()

    if n <= 1:
        return torch.zeros((), device=logits.device)

    pos_logit = logits[pos_idx]

    mask = torch.ones(n, dtype=torch.bool, device=logits.device)
    mask[pos_idx] = False

    neg_logits = logits[mask]

    approx_rank = 1.0 + torch.sigmoid((neg_logits - pos_logit) / tau).sum()
    approx_dcg = 1.0 / torch.log2(approx_rank + 1.0)

    return 1.0 - approx_dcg


def hard_negative_margin_loss(
    logits: torch.Tensor,
    pos_idx: int,
    k: int = HARD_NEG_K,
    margin: float = HARD_NEG_MARGIN,
) -> torch.Tensor:
    n = logits.numel()

    if n <= 1:
        return torch.zeros((), device=logits.device)

    pos_logit = logits[pos_idx]

    mask = torch.ones(n, dtype=torch.bool, device=logits.device)
    mask[pos_idx] = False

    neg_logits = logits[mask]

    if neg_logits.numel() == 0:
        return torch.zeros((), device=logits.device)

    topk = min(k, neg_logits.numel())
    hard_negs = torch.topk(neg_logits, k=topk, largest=True).values

    return F.relu(hard_negs - pos_logit + margin).mean()


def train_mlp_one_epoch(
    model: MLPListwiseRanker,
    X: np.ndarray,
    y: np.ndarray,
    query_slices: Sequence[Tuple[int, int]],
    optimizer: torch.optim.Optimizer,
    rng: np.random.Generator,
) -> float:
    model.train()

    order = np.arange(len(query_slices))
    rng.shuffle(order)

    total_loss = 0.0
    steps = 0

    for batch_start in range(0, len(order), MLP_QUERY_BATCH):
        batch_ids = order[batch_start : batch_start + MLP_QUERY_BATCH]

        row_indices = np.concatenate(
            [
                np.arange(query_slices[i][0], query_slices[i][1], dtype=np.int64)
                for i in batch_ids
            ]
        )

        xb = torch.from_numpy(X[row_indices]).to(DEVICE, non_blocking=True)

        if MLP_INPUT_NOISE_STD > 0:
            xb = xb + torch.randn_like(xb) * MLP_INPUT_NOISE_STD

        if MLP_FEATURE_DROPOUT > 0:
            keep = torch.rand_like(xb) > MLP_FEATURE_DROPOUT
            xb = xb * keep / (1.0 - MLP_FEATURE_DROPOUT)

        logits_all = model(xb)

        losses: List[torch.Tensor] = []
        offset = 0

        for i in batch_ids:
            s, e = query_slices[i]
            n = e - s

            y_slice = y[s:e]
            pos = np.flatnonzero(y_slice > 0)

            if len(pos):
                logits = logits_all[offset : offset + n]
                pos_idx = int(pos[0])

                target = torch.tensor([pos_idx], dtype=torch.long, device=DEVICE)

                ce_loss = F.cross_entropy(logits.unsqueeze(0), target)
                approx_loss = approx_ndcg_loss_from_logits(logits, pos_idx)
                hard_loss = hard_negative_margin_loss(logits, pos_idx)

                mask = torch.ones(n, dtype=torch.bool, device=DEVICE)
                mask[pos_idx] = False
                neg_logits = logits[mask]

                if neg_logits.numel() > 0:
                    pairwise_loss = F.softplus(neg_logits - logits[pos_idx]).mean()
                else:
                    pairwise_loss = torch.zeros((), device=DEVICE)

                if MLP_AUX_BCE_WEIGHT > 0:
                    yb = torch.from_numpy(y_slice.astype(np.float32, copy=False)).to(
                        DEVICE,
                        non_blocking=True,
                    )
                    bce_loss = F.binary_cross_entropy_with_logits(logits, yb)
                else:
                    bce_loss = torch.zeros((), device=DEVICE)

                loss = (
                    ce_loss
                    + MLP_APPROX_NDCG_WEIGHT * approx_loss
                    + MLP_HARD_NEG_WEIGHT * hard_loss
                    + MLP_PAIRWISE_WEIGHT * pairwise_loss
                    + MLP_AUX_BCE_WEIGHT * bce_loss
                )

                losses.append(loss)

            offset += n

        if not losses:
            continue

        loss = torch.stack(losses).mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += float(loss.detach().cpu())
        steps += 1

    return total_loss / max(steps, 1)


@torch.no_grad()
def predict_mlp_scores(
    model: MLPListwiseRanker,
    X: np.ndarray,
    batch_rows: int = 131_072,
) -> np.ndarray:
    model.eval()

    scores = np.empty(len(X), dtype=np.float32)

    for start in range(0, len(X), batch_rows):
        stop = min(start + batch_rows, len(X))

        xb = torch.from_numpy(X[start:stop]).to(DEVICE, non_blocking=True)
        pred = model(xb).detach().cpu().numpy().astype(np.float32)

        scores[start:stop] = pred

    return scores




def _add_query_rank_blend_score(
    frame: pd.DataFrame,
    out_col: str,
    mlp_weight: float,
    rrf_weight: float,
    suffix_weight: float,
    recency_weight: float,
    source_weight: float,
) -> pd.DataFrame:
    """
    Query-wise rank blend used only for selection / final ordering.

    The balanced score keeps MLP as the dominant signal, while RRF and stable
    sequence features act as a small anchor. The safe score is a more conservative
    export generated from the same trained model.
    """
    if frame.empty:
        frame[out_col] = np.array([], dtype=np.float32)
        return frame

    grouped = frame.groupby("qid", sort=False)

    def pct_rank(col: str, default: float = 0.5) -> np.ndarray:
        if col not in frame.columns:
            return np.full(len(frame), default, dtype=np.float32)

        return (
            grouped[col]
            .rank(method="average", pct=True, ascending=True)
            .fillna(default)
            .to_numpy(dtype=np.float32)
        )

    mlp_rank = pct_rank("mlp_score")
    rrf_rank = pct_rank("rrf")
    suffix_rank = pct_rank("suffix_best_prob")
    recency_rank = pct_rank("history_recency")
    source_rank = pct_rank("source_count")

    total = max(
        mlp_weight + rrf_weight + suffix_weight + recency_weight + source_weight,
        1e-12,
    )

    frame[out_col] = (
        (mlp_weight / total) * mlp_rank
        + (rrf_weight / total) * rrf_rank
        + (suffix_weight / total) * suffix_rank
        + (recency_weight / total) * recency_rank
        + (source_weight / total) * source_rank
    ).astype(np.float32)

    return frame


def add_robust_blend_score(
    frame: pd.DataFrame,
    out_col: str = ROBUST_SCORE_COL,
) -> pd.DataFrame:
    return _add_query_rank_blend_score(
        frame,
        out_col=out_col,
        mlp_weight=ROBUST_BLEND_MLP_WEIGHT,
        rrf_weight=ROBUST_BLEND_RRF_WEIGHT,
        suffix_weight=ROBUST_BLEND_SUFFIX_WEIGHT,
        recency_weight=ROBUST_BLEND_RECENCY_WEIGHT,
        source_weight=ROBUST_BLEND_SOURCE_WEIGHT,
    )


def add_safe_blend_score(
    frame: pd.DataFrame,
    out_col: str = SAFE_BLEND_SCORE_COL,
) -> pd.DataFrame:
    return _add_query_rank_blend_score(
        frame,
        out_col=out_col,
        mlp_weight=0.70,
        rrf_weight=0.20,
        suffix_weight=0.04,
        recency_weight=0.04,
        source_weight=0.02,
    )

def report_validation_metrics(
    frame: pd.DataFrame,
    score_col: str = ROBUST_SCORE_COL,
    label: str = "Robust blended ranker",
) -> None:
    valid_recall = frame.groupby("qid", sort=False)["positive_retrieved"].first().mean()
    ndcg, hit = ndcg_hit_from_ranked_frame(frame, score_col)

    print("\nTail meta-validation metrics")
    print(f"  Candidate Recall@{MAX_CANDIDATES}: {valid_recall:.4f}")
    print(f"  {label:<22} Hit@10={hit:.4f}  NDCG@10={ndcg:.4f}")


@dataclass
class RankerBundle:
    mlp_model: MLPListwiseRanker
    feature_cols: List[str]
    rank_cols: List[str]
    scaler_mean: np.ndarray
    scaler_std: np.ndarray
    best_iterations: Dict[str, int]


def fit_mlp_with_validation(
    meta_train: pd.DataFrame,
    meta_valid: pd.DataFrame,
    feature_cols_: Sequence[str],
    rank_cols_: Sequence[str],
) -> Tuple[MLPListwiseRanker, np.ndarray, np.ndarray, int, float]:
    if MLP_TRAIN_NATURAL_ONLY:
        meta_train = natural_recall_subset(meta_train)

    meta_train = meta_train.sort_values("qid", kind="stable").reset_index(drop=True)
    meta_valid = meta_valid.sort_values("qid", kind="stable").reset_index(drop=True)

    print(
        f"\nMLP training queries: {meta_train['qid'].nunique():,}, "
        f"rows: {len(meta_train):,}"
    )

    X_train_raw = mlp_feature_matrix(meta_train, feature_cols_, rank_cols_)
    scaler_mean, scaler_std = fit_standardizer(X_train_raw)
    X_train = standardize_features(X_train_raw, scaler_mean, scaler_std)

    del X_train_raw
    gc.collect()

    X_valid_raw = mlp_feature_matrix(meta_valid, feature_cols_, rank_cols_)
    X_valid = standardize_features(X_valid_raw, scaler_mean, scaler_std)

    del X_valid_raw
    gc.collect()

    y_train = meta_train["label"].astype(np.int8).to_numpy()
    query_slices = make_query_slices(meta_train)

    model = MLPListwiseRanker(input_dim=X_train.shape[1]).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=MLP_LR,
        weight_decay=MLP_WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(MLP_EPOCHS, 1),
        eta_min=MLP_LR * 0.05,
    )

    rng = np.random.default_rng(RANDOM_STATE + 77)

    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 1
    best_ndcg = -1.0
    no_improve = 0

    print(f"\nTraining v4 balanced MLP ranker on {DEVICE}...")

    for epoch in range(1, MLP_EPOCHS + 1):
        loss = train_mlp_one_epoch(
            model=model,
            X=X_train,
            y=y_train,
            query_slices=query_slices,
            optimizer=optimizer,
            rng=rng,
        )

        scheduler.step()

        meta_scored = meta_valid.copy()
        meta_scored["mlp_score"] = predict_mlp_scores(model, X_valid)
        add_robust_blend_score(meta_scored)

        ndcg, hit = ndcg_hit_from_ranked_frame(meta_scored, ROBUST_SCORE_COL)

        print(
            f"  epoch={epoch:02d}  "
            f"loss={loss:.5f}  "
            f"BlendHit@10={hit:.4f}  "
            f"BlendNDCG@10={ndcg:.4f}"
        )

        if ndcg > best_ndcg + 1e-5:
            best_ndcg = ndcg
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        del meta_scored
        gc.collect()

        if no_improve >= MLP_PATIENCE:
            print(f"  early stopping at epoch {epoch}; best epoch={best_epoch}")
            break

    model.load_state_dict(best_state)

    del X_train, X_valid, y_train, query_slices
    gc.collect()

    return model, scaler_mean, scaler_std, best_epoch, best_ndcg


def train_rankers(oof: pd.DataFrame) -> RankerBundle:
    cols = feature_columns(oof)
    rank_cols = selected_mlp_rank_cols(cols)

    print(f"\nBase numeric features after pruning: {len(cols)}")
    print(f"MLP query-rank features: {len(rank_cols)}")
    print(f"Total MLP input dim: {len(cols) + len(rank_cols)}")

    meta_train, meta_valid = split_qids_for_meta_validation(oof)

    meta_train = meta_train.sort_values("qid", kind="stable").reset_index(drop=True)
    meta_valid = meta_valid.sort_values("qid", kind="stable").reset_index(drop=True)

    print(
        f"\nTrue tail validation split: "
        f"train_q={meta_train['qid'].nunique():,}, "
        f"valid_q={meta_valid['qid'].nunique():,}"
    )

    model, mean, std, best_epoch, best_ndcg = fit_mlp_with_validation(
        meta_train,
        meta_valid,
        cols,
        rank_cols,
    )

    X_valid_raw = mlp_feature_matrix(meta_valid, cols, rank_cols)
    X_valid = standardize_features(X_valid_raw, mean, std)

    del X_valid_raw
    gc.collect()

    meta_scored = meta_valid.copy()
    meta_scored["mlp_score"] = predict_mlp_scores(model, X_valid)
    add_robust_blend_score(meta_scored)

    report_validation_metrics(meta_scored, ROBUST_SCORE_COL, "Robust blended ranker")

    print(f"  Selected MLP epochs: {best_epoch}")
    print(f"  Best tail-validation NDCG@10: {best_ndcg:.4f}")

    del model, X_valid, meta_train, meta_valid, meta_scored
    gc.collect()

    print("\nRefitting final v4 MLP on all leakage-free OOF queries...")

    oof_sorted = oof.sort_values("qid", kind="stable").reset_index(drop=True)

    if MLP_TRAIN_NATURAL_ONLY:
        final_train = natural_recall_subset(oof_sorted)
    else:
        final_train = oof_sorted

    final_train = final_train.sort_values("qid", kind="stable").reset_index(drop=True)

    print(
        f"Final MLP training queries: {final_train['qid'].nunique():,}, "
        f"rows: {len(final_train):,}"
    )

    X_all_raw = mlp_feature_matrix(final_train, cols, rank_cols)
    mean_all, std_all = fit_standardizer(X_all_raw)
    X_all = standardize_features(X_all_raw, mean_all, std_all)

    del X_all_raw
    gc.collect()

    y_all = final_train["label"].astype(np.int8).to_numpy()
    query_slices_all = make_query_slices(final_train)

    final_model = MLPListwiseRanker(input_dim=X_all.shape[1]).to(DEVICE)

    optimizer = torch.optim.AdamW(
        final_model.parameters(),
        lr=MLP_LR,
        weight_decay=MLP_WEIGHT_DECAY,
    )

    # Conservative final cap.
    final_epochs = max(4, min(best_epoch, 8))

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(final_epochs, 1),
        eta_min=MLP_LR * 0.05,
    )

    rng = np.random.default_rng(RANDOM_STATE + 177)

    for epoch in range(1, final_epochs + 1):
        loss = train_mlp_one_epoch(
            model=final_model,
            X=X_all,
            y=y_all,
            query_slices=query_slices_all,
            optimizer=optimizer,
            rng=rng,
        )

        scheduler.step()

        print(f"  final epoch={epoch:02d}/{final_epochs}  loss={loss:.5f}")

    del X_all, y_all, query_slices_all, final_train, oof_sorted
    gc.collect()

    return RankerBundle(
        mlp_model=final_model,
        feature_cols=list(cols),
        rank_cols=list(rank_cols),
        scaler_mean=mean_all,
        scaler_std=std_all,
        best_iterations={"mlp_epochs": final_epochs},
    )


# ----------------------------- Test prediction ------------------------------


def predict_test(
    bundle: RankerBundle,
    train: pd.DataFrame,
    test: pd.DataFrame,
    cat_cols: Sequence[str],
) -> Dict[str, Dict[str, List[str]]]:
    print("\nBuilding full-data recall statistics...")
    stats = RecallStats(train, cat_cols)

    print("Generating and ranking test candidates with v4 balanced MLP...")

    predictions: Dict[str, Dict[str, List[str]]] = {
        "balanced": {},
        "safe": {},
        "pure_mlp": {},
    }

    score_cols_by_model = {
        "balanced": ROBUST_SCORE_COL,
        "safe": SAFE_BLEND_SCORE_COL,
        "pure_mlp": "mlp_score",
    }

    columns = list(test.columns)
    batch_size = 600

    for start in range(0, len(test), batch_size):
        stop = min(start + batch_size, len(test))

        batch_rows: List[Dict[str, Any]] = []
        qid_to_uid: Dict[int, str] = {}

        batch = test.iloc[start:stop]

        for local_qid, values in enumerate(batch.itertuples(index=False, name=None)):
            row = dict(zip(columns, values))

            qid = start + local_qid
            uid = clean_id(row.get("uid"))

            qid_to_uid[qid] = uid

            candidates, meta = stats.generate_candidates(row)

            for iid in candidates:
                feats = stats.pair_features(row, iid, meta.get(iid, CandidateMeta()))
                feats["qid"] = qid
                feats["candidate_iid"] = iid
                batch_rows.append(feats)

        cand_frame = pd.DataFrame(batch_rows)

        if cand_frame.empty:
            for _, uid in qid_to_uid.items():
                for model_name in predictions:
                    predictions[model_name][uid] = stats.global_top[:10]
            continue

        X_raw = mlp_feature_matrix(
            cand_frame,
            bundle.feature_cols,
            bundle.rank_cols,
        )

        X = standardize_features(X_raw, bundle.scaler_mean, bundle.scaler_std)

        del X_raw
        gc.collect()

        cand_frame["mlp_score"] = predict_mlp_scores(bundle.mlp_model, X)
        add_robust_blend_score(cand_frame)
        add_safe_blend_score(cand_frame)

        for qid, group in cand_frame.groupby("qid", sort=False):
            uid = qid_to_uid[int(qid)]

            for model_name, score_col in score_cols_by_model.items():
                ranked = (
                    group.sort_values(score_col, ascending=False, kind="stable")[
                        "candidate_iid"
                    ]
                    .drop_duplicates()
                    .tolist()
                )

                top10 = ranked[:10]

                if len(top10) < 10:
                    for iid in stats.global_top:
                        if iid not in top10:
                            top10.append(iid)

                        if len(top10) == 10:
                            break

                predictions[model_name][uid] = top10

        print(f"  predicted {stop:,}/{len(test):,}")

        del cand_frame, X, batch_rows
        gc.collect()

    return predictions


def make_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[str]],
    fallback: Sequence[str],
) -> pd.DataFrame:
    out = sample.copy()
    out["uid"] = out["uid"].map(clean_id)

    out["prediction"] = out["uid"].map(
        lambda uid: ",".join(list(predictions.get(uid, fallback))[:10])
    )

    return out


# ----------------------------------- Main -----------------------------------


def main() -> None:
    seed_everything(RANDOM_STATE)

    print(f"Using device: {DEVICE}")

    print("Loading data...")
    train, test, user, sample = load_data()

    train, test, cat_cols = prepare_frames(train, test, user)

    print(
        f"train={len(train):,}, test={len(test):,}, "
        f"unique targets={train['target_iid'].nunique():,}, "
        f"user cats={len(cat_cols)}"
    )

    print("\nBuilding leakage-free OOF ranking data...")

    oof, fold_recalls = build_oof_training_frame(train, cat_cols)

    print(
        f"\nMean OOF candidate recall@{MAX_CANDIDATES}: "
        f"{np.mean(fold_recalls):.4f} ± {np.std(fold_recalls):.4f}"
    )

    print(f"OOF pair rows: {len(oof):,}; features before pruning: {len(oof.columns)}")
    print(f"MLP usable features after pruning: {len(feature_columns(oof))}")

    bundle = train_rankers(oof)

    print(f"Final selected iterations: {bundle.best_iterations}")

    predictions = predict_test(bundle, train, test, cat_cols)

    full_stats = RecallStats(train, cat_cols)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for model_name, output_name in OUTPUT_NAMES.items():
        submission = make_submission(
            sample,
            predictions[model_name],
            full_stats.global_top[:10],
        )

        output_path = os.path.join(OUTPUT_DIR, output_name)
        submission.to_csv(output_path, index=False)

        print(f"Saved {model_name:>8} submission: {output_path}")

    print("\nRecommended first submission: A2_mlp_v4_balanced.csv")

    print(
        make_submission(
            sample,
            predictions["balanced"],
            full_stats.global_top[:10],
        ).head()
    )


if __name__ == "__main__":
    main()

Using device: cuda
Loading data...
Data directory: /kaggle/input/datasets/theunforgiven7/afac-task2/A�Ƽ�
train=40,000, test=10,000, unique targets=235, user cats=8

Building leakage-free OOF ranking data...

Fold 1/5: fit=32,000, valid=8,000
  leakage-free candidate recall@170: 0.9910

Fold 2/5: fit=32,000, valid=8,000
  leakage-free candidate recall@170: 0.9921

Fold 3/5: fit=32,000, valid=8,000
  leakage-free candidate recall@170: 0.9901

Fold 4/5: fit=32,000, valid=8,000
  leakage-free candidate recall@170: 0.9908

Fold 5/5: fit=32,000, valid=8,000
  leakage-free candidate recall@170: 0.9902

Mean OOF candidate recall@170: 0.9909 ± 0.0007
OOF pair rows: 4,216,543; features before pruning: 103
MLP usable features after pruning: 67

Base numeric features after pruning: 67
MLP query-rank features: 26
Total MLP input dim: 93

True tail validation split: train_q=32,000, valid_q=8,000

MLP training queries: 31,721, rows: 3,343,768

Training v4 balanced MLP ranker on cuda...
  epoch=01  lo